# UDCF aplicado a base Hillstrom — notebook canonico do TCC**Modelo:** UDCF (*Unified Discriminative Causal Forest*), proposto emAi et al., *"LBCF: A Large-Scale Budget-Constrained Causal Forest Algorithm"*, WWW'22.Escopo deste trabalho: apenas a etapa de estimacao de CATE (nao a etapa de otimizacaode orcamento DGB).**Codigo:** C++ original dos autores, clonado em tempo de execucao de`https://github.com/www2022paper/WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS`.Nenhum dos 6 arquivos centrais do algoritmo UDCF e modificado(`UDCFSplittingRule`, `UDCFRelabelingStrategy`, `UDCFPredictionStrategy`,respectivas factories e o scaffolding do GRF). Ver Secao 2 para a lista exatado que e acrescentado.---## Desenho experimentalEste notebook executa quatro etapas, nesta ordem:| Etapa | O que roda | Para que serve ||---|---|---|| **0** | Codigo dos autores nos **dados dos autores** | **Controle.** Prova que o pipeline (clone, compilacao, uso da API) esta correto e que a floresta produz splits quando alimentada com os dados originais. Sem esta etapa, qualquer resultado anomalo depois poderia ser atribuido a erro de configuracao nossa. || **1** | Mesmo binario, **dados Hillstrom**, hiperparametros default dos autores | Resultado principal. A unica coisa que muda em relacao a Etapa 0 sao os dados. || **2** | Diagnostico numerico do criterio de split | Explica *por que* as Etapas 0 e 1 divergem. || **3** | Hillstrom com `imbalance_penalty = 0` | Analise de sensibilidade a um **hiperparametro** (nao ao algoritmo). |A logica do desenho: manter tudo constante e variar uma coisa de cada vez.Etapa 0 -> 1 varia **os dados**. Etapa 1 -> 3 varia **um hiperparametro**.

## 1. Preparacao do ambiente

In [ ]:
!git clone -q https://github.com/www2022paper/WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS.git!apt-get -qq update && apt-get -qq install -y cmake g++

In [ ]:
import osimport zipfileBASE = "WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF"with zipfile.ZipFile(f"{BASE}/LBCF_RCT.zip") as z:    z.extractall(BASE)CORE = os.path.abspath(f"{BASE}/UDCF_RCT/core")BUILD = f"{CORE}/build"DADOS_AUTORES = os.path.abspath(    "WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Data/RCT_data/train_data_UDCF.csv")print("core :", CORE)print("dados dos autores:", DADOS_AUTORES, "| existe?", os.path.exists(DADOS_AUTORES))

## 2. Gerador do `main.cpp``main.cpp` e o **arquivo de entrada do programa**, nao faz parte do algoritmo: eleso carrega dados, diz quais colunas sao outcome/tratamento, chama o treinador egrava as predicoes. Os autores fornecem uma versao propria dele, escrita para osdados deles.**O que esta funcao acrescenta ao `main.cpp` original dos autores** (e exatamente isto,nada mais):1. **Parametrizacao** do caminho dos dados, dos indices de coluna e do numero de   tratamentos — para que o mesmo binario possa rodar nas duas bases.2. **Tres blocos de diagnostico somente-leitura**, que apenas *contam* e *imprimem*   propriedades da floresta ja treinada, sem alterar nada:   - `DIAG_FOREST` — numero de arvores e de variaveis;   - `DIAG_NODES` — total de nos, nos internos (= splits) e quantas arvores sao     tocos (arvores sem nenhum split);   - `DIAG_SPLIT_FREQ` — contagem de splits por variavel, via `SplitFrequencyComputer`,     que ja vem no codigo dos autores.3. **Opcao de sobrescrever `imbalance_penalty`** (usada so na Etapa 3). Quando   `imbalance_penalty=None`, usa-se `ForestTestUtilities::default_options(true, 1)`,   isto e, os defaults dos autores sem nenhum toque.O treinamento em si e sempre `udcf_trainer(num_treatments, 1, true)` — a funcao dosautores, chamada exatamente como eles a chamam.

In [ ]:
def gerar_main_cpp(data_path, outcome_index, treatment_index, num_treatments,                   pred_out, imbalance_penalty=None,                   test_data_path=None, test_outcome_index=None, test_treatment_index=None):    """Gera main.cpp. Ver celula markdown acima para o que e acrescentado ao original."""    trat = ", ".join(str(i) for i in treatment_index)    if imbalance_penalty is None:        bloco_opcoes = (            "    // defaults dos autores, sem nenhuma alteracao\n"            "    ForestOptions options = ForestTestUtilities::default_options(true, 1);"        )        desc_opcoes = "ForestTestUtilities::default_options(true, 1)  [defaults dos autores]"    else:        # Mesmos valores de ForestTestUtilities::default_options(true, 1),        # trocando APENAS imbalance_penalty.        bloco_opcoes = (            "    // identico a ForestTestUtilities::default_options(true, 1),\n"            f"    // exceto imbalance_penalty = {imbalance_penalty}\n"            "    ForestOptions options(300, 1, 0.5, 3, 50, true, 0.5, true, 0.05,\n"            f"                          {imbalance_penalty}, 40, 42, std::vector<size_t>(), 0);"        )        desc_opcoes = f"defaults dos autores com imbalance_penalty = {imbalance_penalty}"    if test_data_path is None:        bloco_pred = (            "    std::vector<Prediction> predictions = predictor.predict_oob(forest, data, false);"        )    else:        trat_teste = ", ".join(str(i) for i in test_treatment_index)        bloco_pred = (            f'    auto data_vec2 = load_data("{test_data_path}");\n'            "    Data data2(data_vec2);\n"            f"    data2.set_outcome_index({test_outcome_index});\n"            f"    data2.set_treatment_index({{{trat_teste}}});\n"            "    std::vector<Prediction> predictions = predictor.predict(forest, data, data2, false);"        )    src = f"""#include <iostream>#include <string>#include <vector>#include <unistd.h>#include "tree/Tree.h"#include "prediction/DefaultPredictionStrategy.h"#include "commons/utility.h"#include "forest/ForestPredictor.h"#include "forest/ForestTrainer.h"#include "utilities/FileTestUtilities.h"#include "utilities/ForestTestUtilities.h"#include "forest/ForestTrainers.h"#include "forest/ForestPredictors.h"#include "analysis/SplitFrequencyComputer.h"using namespace grf;void update_predictions_file(const std::string& file_name,                             const std::vector<Prediction>& predictions) {{  std::vector<std::vector<double>> values;  values.reserve(predictions.size());  for (const auto& prediction : predictions) {{    values.push_back(prediction.get_predictions());  }}  FileTestUtilities::write_csv_file(file_name, values);  std::cout << "predicoes gravadas em " << file_name << std::endl;}}int main(){{    auto data_vec = load_data("{data_path}");    Data data(data_vec);    data.set_outcome_index({outcome_index});    data.set_treatment_index({{{trat}}});    size_t num_treatments = {num_treatments};    ForestTrainer trainer = udcf_trainer(num_treatments, 1, true);{bloco_opcoes}    Forest forest = trainer.train(data, options);    // ---------- diagnostico somente-leitura ----------    std::cout << "DIAG_FOREST num_trees=" << forest.get_trees().size()              << " num_variables=" << forest.get_num_variables() << std::endl;    size_t total_nodes = 0, internal_nodes = 0, stump_trees = 0;    for (const auto& tree : forest.get_trees()) {{      size_t tn = tree->get_child_nodes()[0].size();      size_t internos = 0;      for (size_t n = 0; n < tn; n++) {{        if (!tree->is_leaf(n)) {{          internos++;        }}      }}      total_nodes += tn;      internal_nodes += internos;      if (internos == 0) {{        stump_trees++;      }}    }}    std::cout << "DIAG_NODES total_nodes=" << total_nodes              << " internal_nodes=" << internal_nodes              << " stump_trees=" << stump_trees << std::endl;    SplitFrequencyComputer freq_computer;    std::vector<std::vector<size_t>> freq = freq_computer.compute(forest, 30);    std::vector<size_t> por_var(forest.get_num_variables(), 0);    size_t total_splits = 0;    for (const auto& depth_counts : freq) {{      for (size_t v = 0; v < depth_counts.size(); v++) {{        por_var[v] += depth_counts[v];        total_splits += depth_counts[v];      }}    }}    std::cout << "DIAG_SPLIT_TOTAL " << total_splits << std::endl;    std::cout << "DIAG_SPLIT_FREQ";    for (size_t v = 0; v < por_var.size(); v++) {{      std::cout << " " << v << ":" << por_var[v];    }}    std::cout << std::endl;    // ---------- fim do diagnostico ----------    ForestPredictor predictor = udcf_predictor(1, num_treatments, 1);{bloco_pred}    update_predictions_file("{pred_out}", predictions);    return 0;}}"""    with open(f"{CORE}/main.cpp", "w") as f:        f.write(src)    print("main.cpp gerado")    print("  dados            :", data_path)    print("  outcome_index    :", outcome_index)    print("  treatment_index  :", treatment_index, f"({num_treatments} bracos)")    print("  hiperparametros  :", desc_opcoes)    print("  predicao         :", "predict_oob" if test_data_path is None else "predict (teste separado)")

In [ ]:
import reimport subprocessdef compilar():    os.makedirs(BUILD, exist_ok=True)    r = subprocess.run("cmake .. -DCMAKE_BUILD_TYPE=Release > /dev/null && make -j4",                       shell=True, cwd=BUILD, capture_output=True, text=True)    if r.returncode != 0:        print(r.stdout[-4000:])        print(r.stderr[-4000:])        raise RuntimeError("falha na compilacao")    print("compilado com sucesso")def rodar():    r = subprocess.run("./UDCF_RCT", shell=True, cwd=BUILD, capture_output=True, text=True)    print(r.stdout)    if r.returncode != 0:        print(r.stderr[-4000:])        raise RuntimeError("falha na execucao")    return r.stdoutdef ler_diagnostico(saida):    d = {}    for linha in saida.splitlines():        if linha.startswith(("DIAG_FOREST", "DIAG_NODES")):            for kv in linha.split()[1:]:                k, v = kv.split("=")                d[k] = int(v)        elif linha.startswith("DIAG_SPLIT_TOTAL"):            d["total_splits"] = int(linha.split()[1])        elif linha.startswith("DIAG_SPLIT_FREQ"):            d["splits_por_var"] = {int(p.split(":")[0]): int(p.split(":")[1])                                   for p in linha.split()[1:]}    return ddef resumir(saida, nomes_features=None, titulo=""):    d = ler_diagnostico(saida)    print("=" * 62)    print(f"DIAGNOSTICO — {titulo}")    print("=" * 62)    print(f"  arvores treinadas        : {d['num_trees']}")    print(f"  arvores que sao tocos    : {d['stump_trees']}  "          f"({d['stump_trees'] / d['num_trees'] * 100:.1f}%)")    print(f"  nos internos (= splits)  : {d['internal_nodes']}")    print(f"  total de splits contados : {d['total_splits']}")    if d["total_splits"] == 0:        print("\n  >>> NENHUM SPLIT. A floresta e um conjunto de tocos:")        print("  >>> o CATE estimado e constante (nao ha heterogeneidade modelada).")    else:        print("\n  splits por variavel:")        for v, c in sorted(d["splits_por_var"].items(), key=lambda kv: -kv[1]):            if c == 0:                continue            nome = nomes_features[v] if nomes_features and v < len(nomes_features) else f"col{v}"            print(f"    {nome:<22} {c:>8}  ({c / d['total_splits'] * 100:5.1f}%)")    return d

## Etapa 0 — Controle: codigo dos autores nos dados dos autoresRoda o UDCF na base RCT que acompanha o repositorio, com a **mesma configuracao do`main.cpp` original dos autores**: outcome na coluna 14, tratamento nas colunas 15 a 21(7 bracos), 14 covariaveis (colunas 0 a 13).Ressalva dos proprios autores (`Data/RCT_data/README.md`): sao apenas ~2.000 amostras,criptografadas, disponibilizadas **"NOT FOR REPRODUCE THE RESULT"**. Servem para validaro funcionamento do codigo, nao para reproduzir os numeros do artigo — que e exatamenteo uso que fazemos aqui.**Resultado esperado:** muitos splits. Se isto falhar, o problema esta no nosso pipelinee nada do que vem depois e confiavel.

In [ ]:
gerar_main_cpp(    data_path=DADOS_AUTORES,    outcome_index=14,    treatment_index=[15, 16, 17, 18, 19, 20, 21],    num_treatments=7,    pred_out="predicoes_autores.txt",)compilar()saida_autores = rodar()

In [ ]:
diag_autores = resumir(saida_autores, titulo="ETAPA 0 — dados dos autores")

In [ ]:
import pandas as pdautores = pd.read_csv(DADOS_AUTORES, sep=r"\s+", header=None)y_aut = autores.iloc[:, 14]w_aut = autores.iloc[:, 15:22]print("Caracteristicas da base dos autores")print(f"  dimensoes          : {autores.shape[0]} linhas x {autores.shape[1]} colunas")print(f"  covariaveis        : colunas 0-13 (14 features)")print(f"  bracos             : 7 + controle")print(f"\n  OUTCOME (coluna 14)")print(f"    media            : {y_aut.mean():.2f}")print(f"    desvio padrao    : {y_aut.std():.2f}")print(f"    min / max        : {y_aut.min():.2f} / {y_aut.max():.2f}")print(f"    % de zeros       : {(y_aut == 0).mean() * 100:.1f}%")print(f"    binario?         : {set(y_aut.unique()) <= {0.0, 1.0}}")print(f"\n  tamanho dos grupos:")for k in range(7):    col = 15 + k    print(f"    braco {k + 1} (col {col}): n={int(w_aut[col].sum()):>4}  "          f"Y medio = {y_aut[w_aut[col] == 1].mean():>9.2f}")n_ctrl = int((w_aut.sum(axis=1) == 0).sum())print(f"    controle        : n={n_ctrl:>4}  Y medio = {y_aut[w_aut.sum(axis=1) == 0].mean():>9.2f}")

## Etapa 1 — HillstromBase publica Hillstrom (MineThatData E-Mail Analytics, 2008): experimento aleatorizadode e-mail marketing com 64.000 clientes, 3 bracos (`segment`) e desfecho `conversion`.

In [ ]:
HILLSTROM_URL = (    "http://www.minethatdata.com/"    "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv")df = pd.read_csv(HILLSTROM_URL)print("Base Hillstrom:", df.shape)df.head()

### 1.1 Dicionario de dados| variavel | tipo | descricao ||---|---|---|| `recency` | numerica | meses desde a ultima compra || `history` | numerica | valor gasto (US$) nos 12 meses anteriores ao experimento || `mens` | binaria | comprou roupa masculina no historico || `womens` | binaria | comprou roupa feminina no historico || `newbie` | binaria | cliente novo (cadastro nos ultimos 12 meses) || `zip_code` | categorica | Urban / Surburban / Rural || `channel` | categorica | Phone / Web / Multichannel || `segment` | tratamento | No E-Mail (controle) / Mens E-Mail / Womens E-Mail || `conversion` | desfecho | comprou apos o envio (0/1) — variavel de interesse || `visit`, `spend` | nao utilizadas | pos-tratamento; nao usadas como covariavel (vazamento) |

In [ ]:
print("Tipos de dado:")print(df.dtypes)print("\nResumo estatistico (covariaveis numericas):")print(df[["recency", "history", "mens", "womens", "newbie"]].describe().round(2))print("\nValores faltantes por coluna:")print(df.isna().sum())print("\nLinhas duplicadas:", df.duplicated().sum())

### 1.2 Distribuicoes univariadas`history` apresenta forte assimetria a direita (cauda longa de clientes de alto gasto);as demais covariaveis tem distribuicao proxima da uniforme entre suas categorias,consistente com um desenho experimental balanceado.

In [ ]:
import matplotlib.pyplot as pltfig, axes = plt.subplots(2, 4, figsize=(18, 8))axes[0, 0].hist(df["recency"], bins=12, color="#4C72B0")axes[0, 0].set_title("recency")axes[0, 1].hist(df["history"], bins=40, color="#4C72B0")axes[0, 1].set_title("history")for ax, col in zip([axes[0, 2], axes[0, 3]], ["mens", "womens"]):    df[col].value_counts().sort_index().plot(kind="bar", ax=ax, color="#55A868")    ax.set_title(col)for ax, col in zip([axes[1, 0], axes[1, 1], axes[1, 2]], ["newbie", "zip_code", "channel"]):    df[col].value_counts().sort_index().plot(kind="bar", ax=ax, color="#55A868")    ax.set_title(col)axes[1, 3].axis("off")plt.tight_layout()plt.show()

### 1.3 Checagem de balanceamento entre gruposANOVA para covariaveis continuas, qui-quadrado para categoricas/binarias. Ausencia dediferencas significativas (p > 0,05) corrobora a validade da aleatorizacao — pre-requisitopara interpretacao causal das estimativas.

In [ ]:
from scipy import statslinhas = []for col in ["recency", "history"]:    grupos = [g[col].values for _, g in df.groupby("segment")]    linhas.append((col, "ANOVA", stats.f_oneway(*grupos)[1]))for col in ["mens", "womens", "newbie", "zip_code", "channel"]:    linhas.append((col, "qui-quadrado",                   stats.chi2_contingency(pd.crosstab(df[col], df["segment"]))[1]))print(pd.DataFrame(linhas, columns=["covariavel", "teste", "p_valor"]).round(4).to_string(index=False))

### 1.4 Desfecho: taxa de conversao por grupo (ATE ingenuo)

In [ ]:
taxa = df.groupby("segment")["conversion"].agg(["mean", "count"])print(taxa.round(5))base = taxa.loc["No E-Mail", "mean"]ate_mens = taxa.loc["Mens E-Mail", "mean"] - baseate_womens = taxa.loc["Womens E-Mail", "mean"] - baseprint(f"\nATE ingenuo Mens E-Mail   : {ate_mens:.5f}")print(f"ATE ingenuo Womens E-Mail : {ate_womens:.5f}")print(f"\nComparacao de escala do desfecho:")print(f"  autores  : media = {y_aut.mean():>10.4f} | desvio = {y_aut.std():>10.4f} | continuo")print(f"  Hillstrom: media = {df['conversion'].mean():>10.4f} | desvio = "      f"{df['conversion'].std():>10.4f} | binario")print(f"  razao entre desvios: {y_aut.std() / df['conversion'].std():,.0f}x")

### 1.5 Codificacao e montagem do arquivo de entrada- Desfecho (Y): `conversion`- Tratamento multi-nivel (K=2): `No E-Mail` = controle, `Mens E-Mail` = T1, `Womens E-Mail` = T2- Features: `recency, history, mens, womens, newbie` + one-hot de `zip_code` (3) e `channel` (3) = **11 features**Codificacao one-hot preserva a natureza nominal de `zip_code` e `channel`, evitando imporordem artificial entre categorias sem hierarquia natural. O controle e representado por`T1 = T2 = 0`, mesma convencao usada pelos autores (na base deles, 276 linhas tem as 7colunas de tratamento iguais a zero).

In [ ]:
feature_cols = ["recency", "history", "mens", "womens", "newbie"]X = df[feature_cols].astype(float).copy()X = pd.concat([X,               pd.get_dummies(df["zip_code"], prefix="zip", dtype=float),               pd.get_dummies(df["channel"], prefix="channel", dtype=float)], axis=1)NOMES_FEATURES = list(X.columns)Y = df["conversion"].astype(float)cod = df["segment"].map({"No E-Mail": 0, "Mens E-Mail": 1, "Womens E-Mail": 2})T1 = (cod == 1).astype(float)T2 = (cod == 2).astype(float)design = pd.concat([X, Y.rename("Y"), T1.rename("T1"), T2.rename("T2")], axis=1)n_features = X.shape[1]OUTCOME_IDX = n_featuresTREAT_IDX = [n_features + 1, n_features + 2]ENTRADA_HILLSTROM = f"{CORE}/hillstrom_input.txt"design.to_csv(ENTRADA_HILLSTROM, sep=" ", header=False, index=False)print("features:", NOMES_FEATURES)print("outcome_index:", OUTCOME_IDX, "| treatment_index:", TREAT_IDX)print("arquivo de entrada:", design.shape)

### 1.6 Treinamento — mesmo binario, mesmos hiperparametros, dados diferentes

In [ ]:
gerar_main_cpp(    data_path=ENTRADA_HILLSTROM,    outcome_index=OUTCOME_IDX,    treatment_index=TREAT_IDX,    num_treatments=2,    pred_out="predicoes_hillstrom_default.txt",)compilar()saida_hillstrom = rodar()

In [ ]:
diag_hillstrom = resumir(saida_hillstrom, NOMES_FEATURES,                         titulo="ETAPA 1 — Hillstrom, defaults dos autores")

In [ ]:
def ler_cate(nome_arquivo):    p = pd.read_csv(f"{BUILD}/{nome_arquivo}", header=None, sep=r",\s*", engine="python")    p.columns = ["cate_mens", "cate_womens"]    return pcate_default = ler_cate("predicoes_hillstrom_default.txt")print("=== CATE estimado — Etapa 1 (defaults dos autores) ===")for nome, col, ate in [("Mens E-Mail", "cate_mens", ate_mens),                       ("Womens E-Mail", "cate_womens", ate_womens)]:    c = cate_default[col]    print(f"\n{nome}")    print(f"  media                 : {c.mean():.6f}   (ATE ingenuo: {ate:.6f})")    print(f"  desvio padrao         : {c.std():.6f}")    print(f"  min / max             : {c.min():.6f} / {c.max():.6f}")    print(f"  desvio / media        : {c.std() / abs(c.mean()) * 100:.2f}%")print("\n=== Teste independente: CATE medio por subgrupo ===")print("(se houvesse splits nessas variaveis, os subgrupos teriam que diferir)")aux = df.reset_index(drop=True).join(cate_default)for col in ["zip_code", "channel", "newbie"]:    g = aux.groupby(col)["cate_mens"].mean()    print(f"  {col:<10}: " + ", ".join(f"{k}={v:.6f}" for k, v in g.items()))

## Etapa 2 — Diagnostico: por que as duas bases divergemO criterio de aceite de um split esta em `UDCFSplittingRule.cpp`, linhas ~413-420:```cppdouble decrease = sum_left.square().sum() / weight_sum_left +                  (sum_node - sum_left).square().sum() / weight_sum_right;double penalty_edge = imbalance_penalty * (1.0 / n_left + 1.0 / n_right);decrease -= penalty_edge;if (decrease > 0) { /* candidato aceito */ }```Dois fatos permitem simplificar essa expressao exatamente:1. **Os pesos sao todos 1.** Nenhuma coluna de peso e definida, e `Data::get_weight`   retorna `1.0` nesse caso (`Data.h:165`). Logo `weight_sum_left = n_left`.2. **`sum_node = 0` exatamente.** Os pseudo-desfechos vem do relabeling, que resolve um   OLS de Y sobre W dentro do no. Pela condicao de primeira ordem do OLS, a soma dos   pseudo-desfechos sobre o no e identicamente zero. (Verificado numericamente abaixo.)Com isso, `sum_node - sum_left = -sum_left`, e a expressao inteira fatora:$$\text{decrease} = \left(\frac{1}{n_{esq}} + \frac{1}{n_{dir}}\right)\left(\|s_{esq}\|^2 - \lambda\right)\quad\Longrightarrow\quad\boxed{\;\text{decrease} > 0 \iff \|s_{esq}\|^2 > \lambda\;}$$onde $\lambda$ = `imbalance_penalty` = 0,01 e $s_{esq}$ = soma dos pseudo-desfechos nofilho esquerdo. **O limiar e absoluto**, enquanto o ganho escala com a variancia dodesfecho — o criterio nao e invariante a escala de Y.Se nenhum candidato passa, `find_best_split` retorna `true`(`UDCFSplittingRule.cpp:167-171`) e `TreeTrainer::split_node_internal:238` transforma ono em folha. No no raiz, isso produz um toco.A celula abaixo calcula, para as duas bases, o **maximo atingivel** de $\|s_{esq}\|^2$sobre *todos* os splits possiveis, e compara com o limiar.

In [ ]:
import numpy as npLAMBDA = 0.01MIN_NODE_SIZE = 50ALPHA = 0.05def pseudo_desfechos(Y, W):    """Replica UDCFRelabelingStrategy::relabel (pesos = 1)."""    Yc = Y - Y.mean(axis=0)    Wc = W - W.mean(axis=0)    WW = Wc.T @ Wc    det = np.linalg.det(WW)    A_inv = np.linalg.inv(WW)    beta = A_inv @ Wc.T @ Yc    rho = (Wc @ A_inv.T) * (Yc - Wc @ beta)    return rho, detdef max_norma_sq(Xv, W, rho, respeitar_restricoes=True):    """Maior ||s_esq||^2 sobre todos os splits possiveis."""    n = len(W)    media_w = W.mean(axis=0)    menor = (W < media_w)    n_menor_no = menor.sum(axis=0)    tam_no = (W ** 2).sum(axis=0) - W.sum(axis=0) ** 2 / n    tam_min_filho = tam_no * ALPHA    soma_w, soma_w2 = W.sum(axis=0), (W ** 2).sum(axis=0)    melhor, onde = 0.0, None    for j in range(Xv.shape[1]):        ordem = np.argsort(Xv[:, j], kind="mergesort")        v = Xv[ordem, j]        fronteiras = np.nonzero(np.diff(v) != 0)[0]        if len(fronteiras) == 0:            continue        cum_rho = np.cumsum(rho[ordem], axis=0)[fronteiras]        n_esq = (fronteiras + 1).astype(float)        n_dir = n - n_esq        ok = np.ones(len(fronteiras), dtype=bool)        if respeitar_restricoes:            cum_menor = np.cumsum(menor[ordem], axis=0)[fronteiras]            cum_w = np.cumsum(W[ordem], axis=0)[fronteiras]            cum_w2 = np.cumsum(W[ordem] ** 2, axis=0)[fronteiras]            ok &= (cum_menor >= MIN_NODE_SIZE).all(1)            ok &= ((n_esq[:, None] - cum_menor) >= MIN_NODE_SIZE).all(1)            ok &= ((n_menor_no - cum_menor) >= MIN_NODE_SIZE).all(1)            ok &= ((n_dir[:, None] - n_menor_no + cum_menor) >= MIN_NODE_SIZE).all(1)            var_esq = cum_w2 - cum_w ** 2 / n_esq[:, None]            ok &= (var_esq >= tam_min_filho).all(1)            var_dir = soma_w2 - cum_w2 - (soma_w - cum_w) ** 2 / n_dir[:, None]            ok &= (var_dir >= tam_min_filho).all(1)        if not ok.any():            continue        valores = (cum_rho ** 2).sum(axis=1)        valores = np.where(ok, valores, -np.inf)        i = int(np.argmax(valores))        if valores[i] > melhor:            melhor, onde = float(valores[i]), (j, float(v[fronteiras[i]]))    return melhor, onde# --- base dos autores ---Y_aut = autores.iloc[:, 14].values.reshape(-1, 1).astype(float)W_aut = autores.iloc[:, 15:22].values.astype(float)X_aut = autores.iloc[:, 0:14].values.astype(float)rho_aut, det_aut = pseudo_desfechos(Y_aut, W_aut)max_aut, onde_aut = max_norma_sq(X_aut, W_aut, rho_aut)# --- Hillstrom ---Y_hil = df["conversion"].values.reshape(-1, 1).astype(float)W_hil = np.column_stack([T1.values, T2.values])X_hil = X.values.astype(float)rho_hil, det_hil = pseudo_desfechos(Y_hil, W_hil)max_hil, onde_hil = max_norma_sq(X_hil, W_hil, rho_hil)print("Verificacao de que sum_node = 0 (condicao de 1a ordem do OLS):")print(f"  autores  : maior |soma| = {np.abs(rho_aut.sum(axis=0)).max():.3e}")print(f"  Hillstrom: maior |soma| = {np.abs(rho_hil.sum(axis=0)).max():.3e}")print(f"\n{'':<34}{'AUTORES':>18}{'HILLSTROM':>18}")print("-" * 70)print(f"{'n amostras':<34}{len(autores):>18,}{len(df):>18,}")print(f"{'bracos de tratamento':<34}{7:>18}{2:>18}")print(f"{'desvio padrao do desfecho':<34}{Y_aut.std():>18.4f}{Y_hil.std():>18.4f}")print(f"{'|rho| medio (pseudo-desfechos)':<34}{np.abs(rho_aut).mean():>18.3e}"      f"{np.abs(rho_hil).mean():>18.3e}")print(f"{'max ||s_esq||^2':<34}{max_aut:>18.3e}{max_hil:>18.3e}")print(f"{'limiar (imbalance_penalty)':<34}{LAMBDA:>18.3e}{LAMBDA:>18.3e}")print(f"{'HA SPLIT?':<34}{str(max_aut > LAMBDA):>18}{str(max_hil > LAMBDA):>18}")print("-" * 70)print(f"  autores  : {max_aut / LAMBDA:,.0f}x ACIMA do limiar")print(f"  Hillstrom: {LAMBDA / max_hil:,.0f}x ABAIXO do limiar")print(f"  razao entre as duas bases: {max_aut / max_hil:.2e}")

### 2.1 Confirmando que a causa e a escala do desfechoComo $\rho \propto Y$, tem-se $\|s_{esq}\|^2 \propto c^2$ ao multiplicar Y por umaconstante $c$. O teste abaixo verifica isso diretamente, e tambem testa desfechosalternativos da propria base Hillstrom (`visit` e `spend`), que tem escalas diferentes.Observacao relevante: como $\rho \sim 1/n$, a quantidade $\|s_{esq}\|^2$ *encolhe*conforme n cresce. Sob um limiar absoluto, **mais dados tornam o split mais dificil** —uma propriedade contraintuitiva do criterio, que vale discutir no texto.

In [ ]:
print("Desfechos alternativos da propria base Hillstrom:")for desfecho in ["conversion", "visit", "spend"]:    Yv = df[desfecho].values.reshape(-1, 1).astype(float)    r, _ = pseudo_desfechos(Yv, W_hil)    m, _ = max_norma_sq(X_hil, W_hil, r, respeitar_restricoes=False)    print(f"  {desfecho:<12} desvio={Yv.std():>8.4f}  max||s||^2={m:>11.3e}  "          f"ha split? {m > LAMBDA}")print("\nEscalando artificialmente `conversion` por um fator c:")for c in [1, 10, 50, 100, 1000]:    r, _ = pseudo_desfechos(Y_hil * c, W_hil)    m, _ = max_norma_sq(X_hil, W_hil, r, respeitar_restricoes=False)    print(f"  c = {c:>5}: max||s||^2 = {m:>11.3e}  ha split? {m > LAMBDA}")print("\nEfeito do tamanho da amostra (subamostras de Hillstrom):")rng = np.random.default_rng(42)for n_sub in [2000, 8000, 16000, 32000, 64000]:    idx = rng.choice(len(df), n_sub, replace=False)    r, _ = pseudo_desfechos(Y_hil[idx], W_hil[idx])    m, _ = max_norma_sq(X_hil[idx], W_hil[idx], r, respeitar_restricoes=False)    print(f"  n = {n_sub:>6}: max||s||^2 = {m:>11.3e}")

## Etapa 3 — Sensibilidade: `imbalance_penalty = 0`Pela algebra da Etapa 2, com $\lambda = 0$ o criterio vira$\text{decrease} = \|s_{esq}\|^2 (1/n_{esq} + 1/n_{dir}) > 0$, satisfeito porpraticamente qualquer split.**O que muda aqui:** apenas o valor de `imbalance_penalty` passado ao construtor de`ForestOptions`. Todos os demais hiperparametros permanecem identicos aos defaults dosautores (`num_trees=300`, `sample_fraction=0.5`, `mtry=3`, `min_node_size=50`,`honesty=true`, `honesty_fraction=0.5`, `prune=true`, `alpha=0.05`, `seed=42`).**Nenhum arquivo do algoritmo UDCF e tocado** — `imbalance_penalty` e um hiperparametroexposto pela propria API dos autores.

In [ ]:
gerar_main_cpp(    data_path=ENTRADA_HILLSTROM,    outcome_index=OUTCOME_IDX,    treatment_index=TREAT_IDX,    num_treatments=2,    pred_out="predicoes_hillstrom_ip0.txt",    imbalance_penalty=0.0,)compilar()saida_ip0 = rodar()

In [ ]:
diag_ip0 = resumir(saida_ip0, NOMES_FEATURES,                   titulo="ETAPA 3 — Hillstrom, imbalance_penalty = 0")

In [ ]:
cate_ip0 = ler_cate("predicoes_hillstrom_ip0.txt")print("=== Comparacao: defaults dos autores vs. imbalance_penalty = 0 ===\n")print(f"{'':<26}{'default (0.01)':>18}{'ip = 0':>18}")print("-" * 62)print(f"{'total de splits':<26}{diag_hillstrom['total_splits']:>18,}"      f"{diag_ip0['total_splits']:>18,}")print(f"{'arvores que sao tocos':<26}{diag_hillstrom['stump_trees']:>18}"      f"{diag_ip0['stump_trees']:>18}")for col, nome in [("cate_mens", "CATE mens"), ("cate_womens", "CATE womens")]:    print(f"{nome + ' — media':<26}{cate_default[col].mean():>18.6f}"          f"{cate_ip0[col].mean():>18.6f}")    print(f"{nome + ' — desvio':<26}{cate_default[col].std():>18.6f}"          f"{cate_ip0[col].std():>18.6f}")    print(f"{nome + ' — amplitude':<26}"          f"{cate_default[col].max() - cate_default[col].min():>18.6f}"          f"{cate_ip0[col].max() - cate_ip0[col].min():>18.6f}")saida_final = df.reset_index(drop=True).join(    cate_default.add_suffix("_default")).join(cate_ip0.add_suffix("_ip0"))saida_final.to_csv("hillstrom_cate_resultados.csv", index=False)print("\nArquivo salvo: hillstrom_cate_resultados.csv")

In [ ]:
if diag_ip0["total_splits"] > 0:    fig, axes = plt.subplots(1, 2, figsize=(13, 4))    for ax, col, titulo in [(axes[0], "cate_mens", "Mens E-Mail"),                            (axes[1], "cate_womens", "Womens E-Mail")]:        ax.hist(cate_default[col], bins=60, alpha=0.65, label="default (0.01)", color="#C44E52")        ax.hist(cate_ip0[col], bins=60, alpha=0.65, label="imbalance_penalty = 0", color="#4C72B0")        ax.set_title(f"Distribuicao do CATE — {titulo}")        ax.set_xlabel("CATE estimado")        ax.legend()    plt.tight_layout()    plt.show()else:    print("Etapa 3 tambem nao produziu splits — investigar antes de seguir.")

## SintesePreencher apos a execucao, com os numeros efetivamente obtidos:| | Etapa 0 (autores) | Etapa 1 (Hillstrom, default) | Etapa 3 (Hillstrom, λ=0) ||---|---|---|---|| total de splits | | | || arvores que sao tocos | | | || desvio do CATE | — | | |**Pontos a verificar na leitura dos resultados:**1. A Etapa 0 produziu splits? Se sim, o pipeline esta validado e a Etapa 1 nao pode ser   atribuida a erro de configuracao.2. A Etapa 1 produziu zero splits e 300 tocos? Isso confirma o diagnostico da Etapa 2.3. O CATE medio da Etapa 1 coincide com o ATE ingenuo, e o CATE por subgrupo e constante?   Sao duas confirmacoes independentes da degeneracao, que nao dependem da algebra.4. A Etapa 3 produziu splits? Isso isola `imbalance_penalty` como o fator determinante.**Atencao a interpretacao:** o teste `decrease > 0` **nao e uma modificacao dos autores** —e comportamento original do GRF (ver `InstrumentalSplittingRule.cpp`: inicializa`best_decrease = 0.0` e faz `if (best_decrease <= 0.0) return true;`). O que torna ocriterio restritivo aqui e a combinacao desse teste com `imbalance_penalty = 0.01`(nao-nulo) nos defaults dos autores, aplicada a um desfecho binario raro.